# Local-SDFT: Fine-tune a 230M model on your machine

Self-Distillation Fine-Tuning (**SDFT**) is on-policy self-distillation from demos: show the model a few gold pairs, let it answer, then LoRA-train on those self-generated targets — instead of plain SFT on off-policy gold text. (Shenfeld et al. study this for continual learning / less forgetting; here we run the same loop on a laptop-sized model.)

This Colab contrasts **prompt-only baselines** (no training) and **gold SFT** against the SDFT showcase:

| Arm | What it does |
|-----|----------------|
| **ZS** | Zero-shot base model (inference) |
| **ICL** | Few-shot in-context learning (`FS3` from `sdft.alpacaeval_ablation`) (inference) |
| **CoT** | Chain-of-thought cue (`Let's think step by step.`) (inference) |
| **gold SFT** | LoRA train on `yahma/alpaca-cleaned` gold `output` |
| **SDFT** | Teacher generate → LoRA on self-distilled targets |

**GRPO** is omitted here (slower on free Colab); use the CLI comparison scripts if you want that RL baseline.

Paper: [Shenfeld et al., 2026 — *Self-Distillation Enables Continual Learning*](https://self-distillation.github.io/SDFT) ([arXiv](https://arxiv.org/abs/2601.19897)).

Base model: [Liquid AI LFM2.5-230M](https://huggingface.co/LiquidAI/LFM2.5-230M) — small enough for Colab T4, Apple Silicon MPS, or CPU.

**AlpacaEval-style protocol:** train gold SFT / SDFT on **`yahma/alpaca-cleaned`**, generate on the full **AlpacaEval 2.0** instruction set (805), then pairwise-compare vs `gpt4_turbo` reference. Default judge is a **local** 4-bit open model on T4 (`JUDGE=local`, no API key) → raw `win_rate` + `avg_length` (≈ AE2 protocol, **not** official leaderboard-equivalent). Optional `JUDGE=openai` uses GPT-4-Turbo → `win_rate` / `length_controlled_winrate` (paid). Expect **~1–3+ hours** on Colab T4 for generate+train, plus local-judge time (or OpenAI $ if you switch).



## Setup notes

- **Colab**: Runtime → GPU (T4 is fine). The next cell clones (or **pulls latest `main`**) so `sdft/alpacaeval_local_judge.py` is present, then sets `PYTHONPATH` to the repo root. The install cell uninstalls Colab's stale `torchao` so peft LoRA does not hit a version gate.
- If you see `ModuleNotFoundError: sdft.alpacaeval_local_judge`: **Runtime → Restart session**, then re-run Setup (do not keep a stale clone from before that module landed).
- **Local**: Apple Silicon (MPS) or CUDA preferred; CPU works but is slow on full AE2 generation.
- Expect the working directory to be the **repo root** (`Local-SDFT/`) so `configs/compare/` and `sdft/` resolve.
- **AE2-style win-rate** — train 256 on alpaca-cleaned; generate ZS/ICL/CoT + gold SFT + SDFT on **all 805** AE2 instructions; pairwise judge vs `gpt4_turbo` reference. Default **`JUDGE=local`** (Qwen3.5-9B 4-bit on T4; no OpenAI key). Set **`JUDGE=openai`** + Colab Secret `OPENAI_API_KEY` for official GPT-4-Turbo + LC win-rate (paid). Local judge ≈ AE2 protocol, not leaderboard-equivalent.
- CLI cousin (held-out alpaca heuristic + GRPO, not this notebook’s design): `python scripts/run_batch1_comparison.py --num-train 32 --num-eval 16 --max-grpo-steps 16`.



In [ ]:
import os
import subprocess
import sys
from pathlib import Path

# Install loose pins (Colab / fresh venv).
# peft>=0.19 raises ImportError if torchao is installed but <0.16 (Colab often
# ships 0.10). We don't use torchao quantization for this 230M LoRA path — peft
# falls through to the default nn.Linear dispatcher when torchao is absent.
# bitsandbytes: 4-bit local AE judge on T4 (JUDGE=local).
%pip install -q "torch>=2.6" "transformers>=4.54" "trl>=0.19" "peft>=0.15" "datasets>=2.19" "accelerate>=0.33" "pyyaml>=6.0" "alpaca-eval>=0.6" "bitsandbytes>=0.43"
%pip uninstall -y torchao

REPO_URL = "https://github.com/lin826/Local-SDFT.git"
REPO_DIR = Path("Local-SDFT")
BRANCH = "main"


def _in_colab() -> bool:
    try:
        import google.colab  # noqa: F401  # type: ignore

        return True
    except ImportError:
        return False


def _sync_to_origin_main(root: Path) -> None:
    """Fetch latest main so modules added after an older Colab clone are present."""
    if not (root / ".git").is_dir():
        print(f"WARNING: {root} is not a git checkout; cannot pull latest {BRANCH}")
        return
    print(f"syncing to origin/{BRANCH}…", flush=True)
    subprocess.check_call(
        ["git", "-C", str(root), "fetch", "--depth", "1", "origin", BRANCH]
    )
    judge_py = root / "sdft" / "alpacaeval_local_judge.py"
    # Hard-reset on Colab (ephemeral) or when the local judge module is missing.
    # Keep untracked outputs/; only refresh tracked files. Skip hard reset on a
    # dirty local checkout so WIP is not wiped.
    if _in_colab() or not judge_py.is_file():
        subprocess.check_call(
            ["git", "-C", str(root), "checkout", "-B", BRANCH, f"origin/{BRANCH}"]
        )
    else:
        try:
            subprocess.check_call(
                ["git", "-C", str(root), "pull", "--ff-only", "origin", BRANCH]
            )
        except subprocess.CalledProcessError:
            print(
                "WARNING: could not fast-forward. If imports fail, run:\n"
                f"  git -C {root} checkout -B {BRANCH} origin/{BRANCH}"
            )
    head = subprocess.check_output(
        ["git", "-C", str(root), "rev-parse", "--short", "HEAD"], text=True
    ).strip()
    print(f"HEAD={head} (origin/{BRANCH})", flush=True)


cwd = Path.cwd().resolve()
# Prefer an already-open repo root; else Local-SDFT/ next to cwd; else clone.
if (cwd / "sdft").is_dir() and (cwd / "configs" / "compare").is_dir():
    ROOT = cwd
    print(f"using existing repo root: {ROOT}")
    _sync_to_origin_main(ROOT)
elif (cwd / REPO_DIR / "sdft").is_dir():
    ROOT = (cwd / REPO_DIR).resolve()
    print(f"using cloned repo: {ROOT}")
    _sync_to_origin_main(ROOT)
else:
    subprocess.check_call(
        ["git", "clone", "--depth", "1", "--branch", BRANCH, REPO_URL]
    )
    ROOT = (cwd / REPO_DIR).resolve()
    print(f"cloned into: {ROOT}")

os.chdir(ROOT)
# Ensure imports resolve to this checkout (not a stale sys.path entry).
sys.path = [p for p in sys.path if Path(p).resolve() != ROOT]
sys.path.insert(0, str(ROOT))
os.environ["PYTHONPATH"] = str(ROOT)

judge_py = ROOT / "sdft" / "alpacaeval_local_judge.py"
if not judge_py.is_file():
    raise FileNotFoundError(
        f"Missing {judge_py}. Pull latest main, then re-run this cell:\n"
        f"  git -C {ROOT} fetch origin {BRANCH} && "
        f"git -C {ROOT} checkout -B {BRANCH} origin/{BRANCH}\n"
        "Or: Runtime → Restart session, then re-run Setup."
    )

# Judge mode: local (default, no API key) or openai (GPT-4-Turbo + LC).
# Override: os.environ["JUDGE"] = "openai"  and set OPENAI_API_KEY below.
os.environ.setdefault("JUDGE", "local")
# Optional smaller local id if Qwen3.5-9B OOMs (or older transformers):
# os.environ["ALPACA_EVAL_LOCAL_JUDGE"] = "Qwen/Qwen2.5-7B-Instruct"

# Optional OpenAI key for JUDGE=openai (Colab: Secrets → userdata).
try:
    from google.colab import userdata  # type: ignore

    if not (os.environ.get("OPENAI_API_KEY") or "").strip():
        try:
            os.environ["OPENAI_API_KEY"] = userdata.get("OPENAI_API_KEY")
        except Exception:
            pass
except ImportError:
    pass

try:
    from sdft.alpacaeval_local_judge import default_local_judge_model
    from sdft.alpacaeval_score import resolve_judge_mode
except ModuleNotFoundError as err:
    raise ModuleNotFoundError(
        "Could not import sdft.alpacaeval_local_judge. "
        "Runtime → Restart session, then re-run this Setup cell "
        f"(it syncs origin/{BRANCH}). Do not skip Setup."
    ) from err

JUDGE = resolve_judge_mode()
print(f"JUDGE={JUDGE}")
if JUDGE == "local":
    print(f"local judge model: {default_local_judge_model()}")
    print("(≈ AE2 pairwise protocol; not official GPT-4-Turbo LC leaderboard)")
else:
    from sdft.alpacaeval_score import require_openai_api_key

    try:
        require_openai_api_key()
        print("OPENAI_API_KEY is set (official AlpacaEval annotator ready)")
    except EnvironmentError as err:
        print("ERROR:", err)

print("cwd:", Path.cwd())
print("PYTHONPATH:", os.environ["PYTHONPATH"])
print("judge module:", judge_py)


## Five arms (train on alpaca-cleaned → generate on AE2 → pairwise judge)

```
 yahma/alpaca-cleaned          tatsu-lab/alpaca_eval (805 eval instructions)
        |                                    |
        +---> gold SFT (LoRA on gold output) |
        +---> SDFT (teacher gen → LoRA)      |
        |                                    |
        +---> ZS / ICL / CoT (inference) ----+---> model_outputs JSON per arm
                                             |
                                             v
                                   pairwise vs gpt4_turbo reference
                                   JUDGE=local (default) or openai
                                   → win_rate (+ LC only if openai)
```

- **Train** — gold SFT + SDFT on `yahma/alpaca-cleaned` (not AE2; avoids train/eval leakage).
- **ZS / ICL / CoT** — no LoRA; strategies from `sdft.alpacaeval_ablation` (`ZS`, `FS3`, `CoT`). ICL demos from alpaca-cleaned.
- **Score** — generate on AE2 instructions, then pairwise preference vs `gpt4_turbo` (local open judge by default; optional official `alpaca_eval` GPT-4-Turbo).
- **Dropped for speed:** GRPO only (still available via `scripts/run_batch1_comparison.py`).


In [ ]:
import json
from pathlib import Path

from sdft.alpacaeval_ablation import ALPACA_EVAL_FULL_N, load_alpaca_eval_examples
from sdft.config import DataConfig
from sdft.data import load_examples

# Full run: 256 alpaca-cleaned train + all AE2 (805) generate+judge.
NUM_TRAIN = 256
NUM_EVAL = ALPACA_EVAL_FULL_N  # full AE2 instructions to generate+judge
MAX_JUDGE_INSTANCES = None  # pairwise judge on all generated rows (local or openai)

TRAIN_DATASET = "yahma/alpaca-cleaned"
EVAL_DATASET = "tatsu-lab/alpaca_eval"

train_cfg = DataConfig(
    dataset=TRAIN_DATASET,
    split="train",
    prompt_fields=["instruction", "input"],
    response_field="output",
    num_examples=NUM_TRAIN,
    seed=0,
)
train_ex = load_examples(train_cfg)
assert len(train_ex) == NUM_TRAIN, f"expected {NUM_TRAIN} train rows, got {len(train_ex)}"

eval_ex = load_alpaca_eval_examples(num_examples=NUM_EVAL)
assert len(eval_ex) == NUM_EVAL, f"expected {NUM_EVAL} AE2 rows, got {len(eval_ex)}"

data_dir = Path("data/compare")
data_dir.mkdir(parents=True, exist_ok=True)
gold_path = data_dir / "batch1_gold.jsonl"

with gold_path.open("w", encoding="utf-8") as fh:
    for e in train_ex:
        fh.write(
            json.dumps(
                {
                    "prompt": e["prompt"],
                    "response": e["response"],
                    "sdft_response": e["response"],
                },
                ensure_ascii=False,
            )
            + "\n"
        )

print(
    f"train n={NUM_TRAIN} ({TRAIN_DATASET})  "
    f"eval n={NUM_EVAL}/{ALPACA_EVAL_FULL_N} ({EVAL_DATASET})"
)
print(f"wrote {gold_path} ({NUM_TRAIN} gold SFT rows)")
print("sample train prompt:", train_ex[0]["prompt"][:100].replace("\n", " "), "…")
print("sample AE2 prompt:  ", eval_ex[0]["prompt"][:100].replace("\n", " "), "…")


## SDFT teacher pass

For each **training** prompt (alpaca-cleaned), show the model a few gold demos (`num_shots=2` sampled from the train pool) and let it answer. Those self-generated strings become the SDFT targets.

ZS / ICL / CoT skip this cell (inference-only). AE2 instructions are **not** used here (eval-only).

If you hit CUDA/MPS OOM, drop `max_new_tokens` (e.g. 96) or `generation.batch_size` to 1 in the config / cell below.



In [ ]:
import json
from pathlib import Path

from sdft.config import load_config
from sdft.generate import generate_responses
from sdft.utils import pick_device, release_cuda_memory

sdft_path = Path("data/compare/batch1_sdft.jsonl")
sdft_cfg = load_config("configs/compare/batch1_sdft.yaml")
sdft_cfg.data.num_examples = NUM_TRAIN
sdft_cfg.generation.num_shots = 2
sdft_cfg.generation.out_path = str(sdft_path)
# Safer defaults for Colab T4 / MPS
sdft_cfg.generation.batch_size = 64
sdft_cfg.generation.max_new_tokens = 128

device = pick_device()
print(f"device={device} teacher on {NUM_TRAIN} alpaca-cleaned examples, num_shots=2")

try:
    # generate_responses loads the teacher, generates, then del+empty_cache in finally.
    gens = generate_responses(sdft_cfg, train_ex, device)
except RuntimeError as err:
    if "out of memory" in str(err).lower() or "oom" in str(err).lower():
        raise RuntimeError(
            "OOM during teacher generate. Lower max_new_tokens / batch_size, "
            "or reduce NUM_TRAIN in the data cell."
        ) from err
    raise
finally:
    # Belt-and-suspenders: flush any leftover CUDA cache before LoRA train.
    release_cuda_memory()

# generate_responses returns list[str|None]; tolerate dict rows if already shaped.
rows = []
for ex, gen in zip(train_ex, gens):
    if isinstance(gen, dict):
        row = {
            "prompt": gen.get("prompt", ex["prompt"]),
            "response": gen.get("response", ex["response"]),
            "sdft_response": (gen.get("sdft_response") or "").strip(),
        }
    else:
        text = (gen or "").strip() if isinstance(gen, str) else ""
        row = {
            "prompt": ex["prompt"],
            "response": ex["response"],
            "sdft_response": text,
        }
    if row["sdft_response"]:
        rows.append(row)

sdft_path.parent.mkdir(parents=True, exist_ok=True)
with sdft_path.open("w", encoding="utf-8") as fh:
    for row in rows:
        fh.write(json.dumps(row, ensure_ascii=False) + "\n")

ok = len(rows)
print(f"wrote {sdft_path}  non-empty sdft_response={ok}/{len(gens)}")
if ok == 0:
    raise RuntimeError("teacher pass produced zero non-empty sdft_response rows")


## Train gold SFT + SDFT adapters (`batch_size=16`)

**Gold SFT** and **SDFT** train here on alpaca-cleaned. Repo `configs/compare/` stay at `batch_size=1` for the online-learning demo; this cell writes notebook-local YAML overrides with **`batch_size=16` / `grad_accum=1`** for Colab T4 throughput (not matching the online batch-1 demo). LoRA `r=8` and `max_length=512` are unchanged from those configs.

ZS / ICL / CoT stay inference-only — no adapters. GRPO is not run in this notebook.



In [ ]:
import subprocess
import sys
from pathlib import Path

import yaml

# Colab T4 throughput — do not edit configs/compare/batch1_*.yaml (online batch-1 demo).
TRAIN_BATCH_SIZE = 16
TRAIN_GRAD_ACCUM = 1


def run_mod(*args: str) -> None:
    cmd = [sys.executable, "-m", *args]
    print("+", " ".join(cmd), flush=True)
    subprocess.check_call(cmd, cwd=str(Path.cwd()))


def write_colab_train_cfg(src: str, dst: Path) -> Path:
    """Copy a compare YAML and override training.batch_size / grad_accum for Colab."""
    raw = yaml.safe_load(Path(src).read_text()) or {}
    training = raw.setdefault("training", {})
    training["batch_size"] = TRAIN_BATCH_SIZE
    training["grad_accum"] = TRAIN_GRAD_ACCUM
    dst.parent.mkdir(parents=True, exist_ok=True)
    dst.write_text(yaml.safe_dump(raw, sort_keys=False))
    print(f"wrote {dst} (batch_size={TRAIN_BATCH_SIZE}, grad_accum={TRAIN_GRAD_ACCUM})")
    return dst


gold_path = Path("data/compare/batch1_gold.jsonl")
sdft_path = Path("data/compare/batch1_sdft.jsonl")

out_sft = Path("outputs/compare/batch1-sft-gold")
out_sdft = Path("outputs/compare/batch1-sdft")
colab_cfg_dir = Path("outputs/colab/configs")

gold_cfg = write_colab_train_cfg(
    "configs/compare/batch1_sft_gold.yaml",
    colab_cfg_dir / "batch1_sft_gold_bs16.yaml",
)
sdft_cfg = write_colab_train_cfg(
    "configs/compare/batch1_sdft.yaml",
    colab_cfg_dir / "batch1_sdft_bs16.yaml",
)

run_mod(
    "sdft.train",
    "--config", str(gold_cfg),
    "--data", str(gold_path),
    "--target", "gold",
    "--output-dir", str(out_sft),
)

run_mod(
    "sdft.train",
    "--config", str(sdft_cfg),
    "--data", str(sdft_path),
    "--target", "sdft",
    "--output-dir", str(out_sdft),
)

print("adapters:", out_sft, out_sdft)


## AlpacaEval-style win-rate (local judge default)

Generate on the **AE2 eval** instructions for **zs / icl / cot / sft_gold / sdft**, write `instruction`+`output`+`generator` JSON, then pairwise-compare each output to the **gpt4_turbo** reference.

- **`JUDGE=local`** (default): 4-bit `Qwen/Qwen3.5-9B` (override via `ALPACA_EVAL_LOCAL_JUDGE`). Metrics: **`win_rate`**, **`avg_length`**, **`standard_error`**. `length_controlled_winrate` is **not** available (OpenAI GLM pipeline only). Unload the 230M policy first (this cell empties CUDA cache after generation).
- **`JUDGE=openai`**: official [`alpaca_eval.evaluate`](https://github.com/tatsu-lab/alpaca_eval) annotator `weighted_alpaca_eval_gpt4_turbo` → **`win_rate`** + **`length_controlled_winrate`** (needs `OPENAI_API_KEY`; paid).

Local judge ≈ AE2 protocol shape — **not** official leaderboard-equivalent.


In [ ]:
import os
import re
from pathlib import Path

import torch
from peft import PeftModel

from sdft.alpacaeval_ablation import (
    build_eval_messages,
    get_ablation_arm,
    web_few_shots,
)
from sdft.alpacaeval_score import (
    evaluate_model_outputs,
    resolve_judge_mode,
    to_model_outputs,
    write_model_outputs,
)
from sdft.config import ModelConfig, load_config
from sdft.peft_utils import adapter_ready
from sdft.utils import load_model, load_tokenizer, pick_device, release_cuda_memory, to_model_device

try:
    from sdft.alpacaeval_local_judge import default_local_judge_model
except ModuleNotFoundError as err:
    raise ModuleNotFoundError(
        "Missing sdft.alpacaeval_local_judge. "
        "Runtime → Restart session, then re-run the Setup cell "
        "(it git-syncs latest main and sets PYTHONPATH). Do not skip Setup."
    ) from err

JUDGE = resolve_judge_mode()
if JUDGE == "openai":
    from sdft.alpacaeval_score import require_openai_api_key

    require_openai_api_key()
else:
    print(f"using local judge: {default_local_judge_model()}", flush=True)

REFUSAL_RE = re.compile(r"(?i)\b(i('m| am) sorry|i can('t|not) (assist|help)|as an ai)\b")

MODEL_NAME = load_config("configs/compare/batch1_sdft.yaml").model.name  # LiquidAI/LFM2.5-230M
eval_prompts = [e["prompt"] for e in eval_ex]

PROMPT_ARMS = {
    "zs": get_ablation_arm("ZS"),
    "icl": get_ablation_arm("FS3"),
    "cot": get_ablation_arm("CoT"),
}
ICL_DEMOS = web_few_shots(PROMPT_ARMS["icl"].few_shot_k, seed=0)
SFT_GOLD_ADAPTER = Path("outputs/compare/batch1-sft-gold")
SDFT_ADAPTER = Path("outputs/compare/batch1-sdft")
ADAPTER_ARMS = {
    "sft_gold": SFT_GOLD_ADAPTER,
    "sdft": SDFT_ADAPTER,
}
ARM_ORDER = ["zs", "icl", "cot", "sft_gold", "sdft"]
OUT_ROOT = Path("outputs/alpacaeval")

print(
    f"AE2 generate+judge n={len(eval_prompts)}  "
    f"max_instances={MAX_JUDGE_INSTANCES} (None = full)",
    flush=True,
)
print(f"ICL demos (FS3): {len(ICL_DEMOS)} from alpaca-cleaned", flush=True)


@torch.inference_mode()
def generate_prompt_arms(max_new_tokens: int = 128) -> dict[str, list[str]]:
    device = pick_device()
    cfg = ModelConfig(name=MODEL_NAME)
    tokenizer = load_tokenizer(cfg)
    tokenizer.padding_side = "left"
    model = load_model(cfg, device)
    model.eval()
    outs: dict[str, list[str]] = {name: [] for name in PROMPT_ARMS}
    for i, prompt in enumerate(eval_prompts):
        if (i + 1) % 50 == 0 or i == 0:
            print(f"  prompt-arms gen {i + 1}/{len(eval_prompts)}…", flush=True)
        for name, settings in PROMPT_ARMS.items():
            demos = ICL_DEMOS if settings.few_shot_k > 0 else None
            messages = build_eval_messages(prompt, settings, few_shots=demos)
            text = tokenizer.apply_chat_template(
                messages, tokenize=False, add_generation_prompt=True
            )
            enc = to_model_device(
                tokenizer(text, return_tensors="pt", add_special_tokens=False),
                model,
            )
            out = model.generate(
                **enc,
                max_new_tokens=max_new_tokens,
                do_sample=False,
                pad_token_id=tokenizer.pad_token_id,
            )
            new = out[:, enc["input_ids"].shape[1] :]
            outs[name].append(tokenizer.decode(new[0], skip_special_tokens=True).strip())
    del model, tokenizer
    release_cuda_memory()
    return outs


@torch.inference_mode()
def generate_adapter(adapter: Path, arm: str, max_new_tokens: int = 128) -> list[str]:
    device = pick_device()
    cfg = ModelConfig(name=MODEL_NAME)
    tokenizer = load_tokenizer(cfg)
    tokenizer.padding_side = "left"
    base = load_model(cfg, device)
    model = PeftModel.from_pretrained(base, str(adapter))
    model.eval()
    settings = PROMPT_ARMS["zs"]
    outs: list[str] = []
    for i, prompt in enumerate(eval_prompts):
        if (i + 1) % 50 == 0 or i == 0:
            print(f"  {arm} gen {i + 1}/{len(eval_prompts)}…", flush=True)
        messages = build_eval_messages(prompt, settings, few_shots=None)
        text = tokenizer.apply_chat_template(
            messages, tokenize=False, add_generation_prompt=True
        )
        enc = to_model_device(
            tokenizer(text, return_tensors="pt", add_special_tokens=False),
            model,
        )
        out = model.generate(
            **enc,
            max_new_tokens=max_new_tokens,
            do_sample=False,
            pad_token_id=tokenizer.pad_token_id,
        )
        new = out[:, enc["input_ids"].shape[1] :]
        outs.append(tokenizer.decode(new[0], skip_special_tokens=True).strip())
    del model, base, tokenizer
    release_cuda_memory()
    return outs


arm_results: dict[str, dict] = {}
ae_rows = []

print("generating zs / icl / cot on AE2…", flush=True)
prompt_gens = generate_prompt_arms()
for name, gens in prompt_gens.items():
    rows = to_model_outputs(eval_prompts, gens, generator=name)
    write_model_outputs(OUT_ROOT / name / "model_outputs.json", rows)
    arm_results[name] = {
        "gens": gens,
        "refusals": [bool(REFUSAL_RE.search(g)) for g in gens],
        "adapter": None,
        "rows": rows,
    }

for name, adapter in ADAPTER_ARMS.items():
    if adapter_ready(adapter):
        print(f"generating {name} on AE2…", flush=True)
        gens = generate_adapter(adapter, name)
        rows = to_model_outputs(eval_prompts, gens, generator=name)
        write_model_outputs(OUT_ROOT / name / "model_outputs.json", rows)
        arm_results[name] = {
            "gens": gens,
            "refusals": [bool(REFUSAL_RE.search(g)) for g in gens],
            "adapter": adapter,
            "rows": rows,
        }
    else:
        print(f"skip {name}: adapter missing at {adapter}")

print(f"running {JUDGE} pairwise judge…", flush=True)
for name in ARM_ORDER:
    if name not in arm_results:
        continue
    print(f"  judging {name}…", flush=True)
    summary = evaluate_model_outputs(
        arm_results[name]["rows"],
        name=name,
        output_dir=OUT_ROOT / name,
        max_instances=MAX_JUDGE_INSTANCES,
        judge=JUDGE,
        judge_model=os.environ.get("ALPACA_EVAL_LOCAL_JUDGE") or None,
    )
    m = summary["metrics"]
    arm_results[name]["metrics"] = m
    ae_rows.append(
        {
            "arm": name,
            "n": m.get("n_total") or len(arm_results[name]["gens"]),
            "win_rate": m.get("win_rate"),
            "length_controlled_winrate": m.get("length_controlled_winrate"),
            "standard_error": m.get("standard_error"),
            "avg_length": m.get("avg_length"),
            "refusal_rate": round(
                sum(arm_results[name]["refusals"]) / max(len(arm_results[name]["gens"]), 1),
                3,
            ),
        }
    )

print()
print("| arm | n | win_rate | length_controlled_winrate | std_err | avg_length | refusal_rate |")
print("|-----|---|----------|---------------------------|---------|------------|--------------|")
for r in ae_rows:
    wr = r["win_rate"]
    lc = r["length_controlled_winrate"]
    se = r["standard_error"]
    al = r["avg_length"]
    print(
        f"| {r['arm']} | {r['n']} | "
        f"{wr if wr is None else f'{wr:.2f}'} | "
        f"{lc if lc is None else f'{lc:.2f}'} | "
        f"{se if se is None else f'{se:.2f}'} | "
        f"{al if al is None else f'{al:.1f}'} | "
        f"{r['refusal_rate']:.3f} |"
    )
print()
if JUDGE == "local":
    print(
        f"metrics = local open judge ({default_local_judge_model()}); "
        f"win_rate + avg_length (LC N/A); wrote under {OUT_ROOT}/"
    )
else:
    print(
        f"metrics = official AlpacaEval (annotator weighted_alpaca_eval_gpt4_turbo); "
        f"wrote under {OUT_ROOT}/"
    )


## Qualitative sample — same AE2 prompt, all arms

Aggregate win-rates can look flat while one prompt shows **ZS refusing** and **SDFT answering**. This cell picks one AE2 eval prompt (preferring an SDFT non-refusal vs a ZS refusal when available) and shows side-by-side outputs. Optional / illustrative — primary metric is official LC / win-rate above.



In [ ]:
# Qualitative: same AE2 prompt across arms (no instruction_reward aggregate).
import html
import re
from pathlib import Path

from IPython.display import HTML, display

REFUSAL_RE = re.compile(r"(?i)\b(i('m| am) sorry|i can('t|not) (assist|help)|as an ai)\b")
TRUNCATE_AT = 600
FALLBACK_PROMPTS = [
    "How do I sew a button on a shirt?",
    "How do I make apple juice?",
]
ARM_ORDER = ["zs", "icl", "cot", "sft_gold", "sdft"]


def truncate_display(text: str, limit: int = TRUNCATE_AT) -> tuple[str, str]:
    if len(text) <= limit:
        return text, ""
    note = f" [truncated; full length={len(text)} chars]"
    return text[:limit] + "…", note


results = globals().get("arm_results") or {}
prompts_list = list(globals().get("eval_prompts") or [e["prompt"] for e in eval_ex])
n = len(prompts_list)

idx = None
pick_reason = "no arm_results — run the official score cell first"
if results and "sdft" in results and "zs" in results:
    for i in range(n):
        zs_r = bool(results["zs"]["refusals"][i])
        sd_r = bool(results["sdft"]["refusals"][i])
        if zs_r and not sd_r:
            idx, pick_reason = i, "ZS refuses, SDFT answers (AE2 eval index)"
            break
    if idx is None:
        for i in range(n):
            if not bool(results["sdft"]["refusals"][i]):
                idx, pick_reason = i, "first non-refusing SDFT row on AE2 eval"
                break
    if idx is None:
        idx, pick_reason = 0, "fallback to AE2 index 0"

if idx is not None and results:
    prompt_used = prompts_list[idx]
    arm_outputs = {}
    for arm in ARM_ORDER:
        if arm not in results:
            continue
        gen = results[arm]["gens"][idx]
        arm_outputs[arm] = {
            "output": gen,
            "refusal": bool(results[arm]["refusals"][idx]),
            "lc": (results[arm].get("metrics") or {}).get("length_controlled_winrate"),
        }
else:
    prompt_used = FALLBACK_PROMPTS[0]
    arm_outputs = {}
    pick_reason = f"README fallback prompt (no scored gens): {prompt_used!r}"

print(f"reason={pick_reason}")
print(f"prompt={prompt_used!r}")

header = (
    "<h3>Same-prompt comparison (AE2)</h3>"
    f"<p><b>Selection:</b> {html.escape(pick_reason)}</p>"
    f"<p><b>Prompt:</b> {html.escape(prompt_used)}</p>"
)
rows_html = [
    "<table style='border-collapse:collapse;width:100%;font-size:13px'>",
    "<tr>"
    "<th style='border:1px solid #ccc;padding:6px;text-align:left'>arm</th>"
    "<th style='border:1px solid #ccc;padding:6px;text-align:left'>refusal</th>"
    "<th style='border:1px solid #ccc;padding:6px;text-align:left'>output</th>"
    "</tr>",
]
for arm in ARM_ORDER:
    if arm not in arm_outputs:
        continue
    o = arm_outputs[arm]
    shown, note = truncate_display(o["output"])
    refuse_s = "yes" if o["refusal"] else "no"
    rows_html.append(
        "<tr>"
        f"<td style='border:1px solid #ccc;padding:6px;vertical-align:top'><b>{html.escape(arm)}</b></td>"
        f"<td style='border:1px solid #ccc;padding:6px;vertical-align:top'>{refuse_s}</td>"
        f"<td style='border:1px solid #ccc;padding:6px;vertical-align:top;"
        f"white-space:pre-wrap;font-family:ui-monospace,monospace'>"
        f"{html.escape(shown)}{html.escape(note)}</td>"
        "</tr>"
    )
rows_html.append("</table>")
display(HTML(header + "".join(rows_html)))

print("| arm | refusal | output (truncated) |")
print("|-----|---------|-------------------|")
for arm in ARM_ORDER:
    if arm not in arm_outputs:
        continue
    o = arm_outputs[arm]
    shown, note = truncate_display(o["output"], limit=160)
    flat = (shown + note).replace("|", "\\|").replace("\n", " ")
    print(f"| {arm} | {'yes' if o['refusal'] else 'no'} | {flat} |")


## What to try next

1. **Switch judge** — `JUDGE=local` (default, free on T4) or `JUDGE=openai` + Secret `OPENAI_API_KEY` for official LC win-rate.
2. **Swap local judge size** — `ALPACA_EVAL_LOCAL_JUDGE=Qwen/Qwen2.5-7B-Instruct` if 9B OOMs.
3. **Re-judge only** — `python scripts/run_alpaca_eval.py --model-outputs outputs/alpacaeval/sdft/model_outputs.json --name sdft`.
4. **Swap ICL depth** — change `get_ablation_arm("FS3")` to `"FS1"` in the score cell.
5. **Online learning web UI** — locally: `python -m web.app`, open `/data`.
6. **Held-out batch-1 CLI** (heuristic + GRPO) — `python scripts/run_batch1_comparison.py --num-train 32 --num-eval 16 --max-grpo-steps 16`.

Happy tinkering — 230M is small enough that curiosity is the bottleneck, not VRAM.
